<a href="https://colab.research.google.com/github/Yipweiyang/SIT-UofG-QC-Assignment/blob/main/BB84-Plain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 79.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=6bfce8d531529d029bdda0d2d8d2139c2efde8f3018561e30c7373dbb32dacfb
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



In [2]:
# Quantum random bit generator
# This is used instead of Python's random module.

def quantum_random_bit():
    qc = QuantumCircuit(1, 1)

    # Put qubit into superposition: (|0> + |1>) / sqrt(2)
    qc.h(0)

    # Measure the qubit
    qc.measure(0, 0)

    simulator = BasicSimulator()
    result = simulator.run(qc, shots=1).result()
    counts = result.get_counts()

    measured_bit = list(counts.keys())[0]
    return int(measured_bit)

In [3]:
# Test the quantum random bit generator

for i in range(10):
    print(quantum_random_bit())

0
1
0
0
1
0
0
1
1
0


In [4]:
# Alice generates random bits and random bases

n = 20  # number of qubits to send

alice_bits = []
alice_bases = []

for i in range(n):
    alice_bits.append(quantum_random_bit())

    if quantum_random_bit() == 0:
        alice_bases.append("s")  # standard basis
    else:
        alice_bases.append("d")  # diagonal basis

print("Alice bits: ", alice_bits)
print("Alice bases:", alice_bases)

Alice bits:  [0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1]
Alice bases: ['d', 's', 's', 'd', 'd', 's', 'd', 'd', 'd', 's', 's', 's', 'd', 'd', 'd', 'd', 's', 's', 's', 'd']


In [5]:
# Alice prepares qubits

alice_qubits = []

for i in range(n):

    qc = QuantumCircuit(1, 1)

    bit = alice_bits[i]
    basis = alice_bases[i]

    # If bit is 1, apply X gate
    if bit == 1:
        qc.x(0)

    # If basis is diagonal, apply H gate
    if basis == "d":
        qc.h(0)

    alice_qubits.append(qc)

print("Alice prepared", len(alice_qubits), "qubits.")

Alice prepared 20 qubits.


In [6]:
# Bob chooses random measurement bases

bob_bases = []

for i in range(n):
    if quantum_random_bit() == 0:
        bob_bases.append("s")  # standard basis
    else:
        bob_bases.append("d")  # diagonal basis

print("Bob bases:", bob_bases)

Bob bases: ['s', 'd', 's', 'd', 's', 'd', 'd', 's', 'd', 'd', 'd', 's', 's', 's', 'd', 'd', 's', 'd', 'd', 'd']


In [7]:
# Bob measures the qubits

bob_bits = []

simulator = BasicSimulator()

for i in range(n):

    qc = alice_qubits[i].copy()

    # If Bob uses diagonal basis,
    # apply H before measurement
    if bob_bases[i] == "d":
        qc.h(0)

    qc.measure(0, 0)

    result = simulator.run(qc, shots=1).result()
    counts = result.get_counts()

    measured_bit = int(list(counts.keys())[0])

    bob_bits.append(measured_bit)

print("Bob bits:", bob_bits)

Bob bits: [1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1]


In [8]:
# Alice and Bob compare bases publicly

matching_indices = []

for i in range(n):
    if alice_bases[i] == bob_bases[i]:
        matching_indices.append(i)

print("Matching indices:", matching_indices)
print("Number of matching bases:", len(matching_indices))

Matching indices: [2, 3, 6, 8, 11, 14, 15, 16, 19]
Number of matching bases: 9


In [9]:
# Alice and Bob keep only the bits where their bases matched

alice_key = []
bob_key = []

for i in matching_indices:
    alice_key.append(alice_bits[i])
    bob_key.append(bob_bits[i])

print("Alice key:", alice_key)
print("Bob key:  ", bob_key)
print("Keys match:", alice_key == bob_key)

Alice key: [0, 1, 1, 0, 1, 0, 0, 1, 1]
Bob key:   [0, 1, 1, 0, 1, 0, 0, 1, 1]
Keys match: True


In [10]:
# Final summary for BB84 without attacker

print("BB84 without attacker")
print("---------------------")
print("Number of qubits sent:", n)
print("Number of matching bases:", len(matching_indices))
print("Alice key:", alice_key)
print("Bob key:  ", bob_key)
print("Keys match:", alice_key == bob_key)

if alice_key == bob_key:
    print("Result: Successful key exchange. No attacker detected.")
else:
    print("Result: Keys do not match. Something went wrong.")

BB84 without attacker
---------------------
Number of qubits sent: 20
Number of matching bases: 9
Alice key: [0, 1, 1, 0, 1, 0, 0, 1, 1]
Bob key:   [0, 1, 1, 0, 1, 0, 0, 1, 1]
Keys match: True
Result: Successful key exchange. No attacker detected.
